In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Clustering_Project") \
    .config("spark.executor.memory", "8g") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.caseSensitive", "true") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .getOrCreate()

df = spark.read.json("cleaned_review.json")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/11 20:57:51 WARN Utils: Your hostname, Pangkaews-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.32.148.207 instead (on interface en0)
26/03/11 20:57:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/11 20:57:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/11 20:57:52 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/11 20:57:52 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/03/11 20:57:52 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [3]:


from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans, BisectingKMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml import Pipeline
import matplotlib.pyplot as plt


In [4]:
from pyspark.sql import functions as F

# create season column
df = df.withColumn(
    "season",
    F.when(F.col("month").isin(12, 1, 2), "Winter")
     .when(F.col("month").isin(3, 4, 5), "Spring")
     .when(F.col("month").isin(6, 7, 8), "Summer")
     .otherwise("Autumn")
)

# can be changed
curr_date = F.to_date(F.lit("2024-01-01"))

user_features = df.groupBy("user_id").agg(
    F.datediff(curr_date, F.max("timestamp_convert")).alias("recency_days"),
    F.avg(F.when(F.col("verified_purchase") == True, 1.0).otherwise(0.0)).alias("verified_rate"),
    F.sum(F.when(F.col("season") == "Summer", 1).otherwise(0)).alias("summer_count"),
    F.sum(F.when(F.col("season") == "Winter", 1).otherwise(0)).alias("winter_count"),
    F.sum(F.when(F.col("season") == "Spring", 1).otherwise(0)).alias("spring_count"),
    F.sum(F.when(F.col("season") == "Autumn", 1).otherwise(0)).alias("autumn_count"),
    F.count("user_id").alias("total_activity")
)

# 60% can be changed
user_features = user_features.withColumn("is_summer_shopper", 
    F.when((F.col("summer_count") / F.col("total_activity")) > 0.6, 1.0).otherwise(0.0))
user_features = user_features.withColumn("is_winter_shopper", 
    F.when((F.col("winter_count") / F.col("total_activity")) > 0.6, 1.0).otherwise(0.0))
user_features = user_features.withColumn("is_spring_shopper", 
    F.when((F.col("spring_count") / F.col("total_activity")) > 0.6, 1.0).otherwise(0.0))
user_features = user_features.withColumn("is_autumn_shopper", 
    F.when((F.col("autumn_count") / F.col("total_activity")) > 0.6, 1.0).otherwise(0.0))



In [5]:
# vectorisation

from pyspark.ml.feature import VectorAssembler, StandardScaler

feature_cols = ["recency_days", "verified_rate", "is_summer_shopper", "is_winter_shopper", "is_autumn_shopper", "is_spring_shopper"]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
#scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=True)
scaler = StandardScaler(inputCol="raw_features", outputCol="features")
pipeline = Pipeline(stages=[assembler, scaler])

final_df = pipeline.fit(user_features).transform(user_features)

In [6]:
# clustering

#find k

from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator = ClusteringEvaluator(featuresCol="features")

results = []

for k in range(2, 11):
    kmeans = KMeans(featuresCol="features", k=k, seed= 42)
    model = kmeans.fit(final_df)
    predictions = model.transform(final_df)
    cost = model.summary.trainingCost
    silhouette = evaluator.evaluate(predictions)
    
    results.append((k, cost, silhouette))

print(f"result : \n {results}")


26/03/11 20:57:58 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/03/11 20:58:02 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


result : 
 [(2, 2967351.9233512687, 0.3683673708313325), (3, 2165215.4595152736, 0.5418931773775115), (4, 1375176.560388545, 0.7144501776698964), (5, 916014.6615536598, 0.7805497762130374), (6, 811363.88728662, 0.7312199469993937), (7, 755270.5324186139, 0.7516557477519928), (8, 663968.6702611893, 0.7422805828008304), (9, 609513.6190149292, 0.7281221047946732), (10, 554767.1298009936, 0.7483839778218903)]


In [7]:
# select k=5 because the maximum sillouete score
kmeans = KMeans(featuresCol="features", k=5, seed= 42)
model = kmeans.fit(final_df)
predictions = model.transform(final_df)




In [9]:
# representative

profile = predictions.groupBy("prediction").agg(
    F.count("user_id").alias("Cluster_Size"),
    F.avg("recency_days").alias("Avg_Recency"),
    F.avg("verified_rate").alias("Avg_Verified_Rate"),
    F.round(F.avg("is_summer_shopper"),2).alias("Pct_Summer_Shoppers"),
    F.round(F.avg("is_winter_shopper"),2).alias("Pct_Winter_Shoppers"),
    F.round(F.avg("is_autumn_shopper"),2).alias("Pct_Autumn_Shoppers"),
    F.round(F.avg("is_spring_shopper"),2).alias("Pct_Spring_Shoppers"),
)
profile.show()



+----------+------------+------------------+--------------------+-------------------+-------------------+-------------------+-------------------+
|prediction|Cluster_Size|       Avg_Recency|   Avg_Verified_Rate|Pct_Summer_Shoppers|Pct_Winter_Shoppers|Pct_Autumn_Shoppers|Pct_Spring_Shoppers|
+----------+------------+------------------+--------------------+-------------------+-------------------+-------------------+-------------------+
|         1|      168128| 1670.830224590788|  0.9983820478923908|                0.0|               0.89|                0.0|                0.0|
|         3|      126889|1683.7198575132595|  0.9985196941909107|                0.0|                0.0|                1.0|                0.0|
|         4|       50082|2157.0970608202547|0.024183308154541415|               0.23|               0.24|               0.23|               0.21|
|         2|      142949|1694.3857179833367|  0.9987728390644813|                0.0|                0.0|                0.0

K-mean for customer loyalty

In [10]:

user_loyalty = df.groupBy("user_id").agg(
    F.avg("rating").alias("avg_rating"),
    F.avg(F.length("text")).alias("avg_review_length"),
    F.count("user_id").alias("total_reviews")
)

user_loyalty = user_loyalty.withColumn("sentiment_score", 
    F.when(F.col("avg_rating") >= 4, 1.0)
     .when(F.col("avg_rating") <= 2, -1.0)
     .otherwise(0.0)
)



In [12]:
feature_cols = ["sentiment_score", "avg_review_length","total_reviews"]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=True)

pipeline = Pipeline(stages=[assembler, scaler])
user_final_df = pipeline.fit(user_loyalty).transform(user_loyalty)

In [13]:


evaluator = ClusteringEvaluator(featuresCol="features")

results = []

for k in range(2, 11):
    kmeans = KMeans(featuresCol="features", k=k, seed= 42)
    model = kmeans.fit(user_final_df)
    predictions_user_findk = model.transform(user_final_df)
    cost = model.summary.trainingCost
    silhouette = evaluator.evaluate(predictions_user_findk)
    
    results.append((k, cost, silhouette))

print(f"result : \n {results}")

result : 
 [(2, 1627452.1696887384, 0.9983072830188092), (3, 1055590.1693640382, 0.6515529579028558), (4, 745558.5100310127, 0.7344270826045198), (5, 643543.914198738, 0.7508267541162814), (6, 547991.5308441028, 0.7659419322310344), (7, 528970.6266109998, 0.6909245868860499), (8, 429271.734572773, 0.6814871373315404), (9, 350214.9071713633, 0.7422877917375245), (10, 384841.0110152147, 0.713797362597145)]


In [14]:
kmeans = KMeans(featuresCol="features", k=2, seed= 42)
model = kmeans.fit(user_final_df)
predictions_user = model.transform(user_final_df)

In [15]:
profile_user = predictions_user.groupBy("prediction").agg(
    F.count("user_id").alias("Cluster_Size"),
    F.avg("sentiment_score").alias("Avg_sentiment_score"),
    F.avg("avg_review_length").alias("Avg_review_length"),
    F.round(F.avg("total_reviews"),2).alias("total_review")
)

profile_user.show()

+----------+------------+-------------------+-----------------+------------+
|prediction|Cluster_Size|Avg_sentiment_score|Avg_review_length|total_review|
+----------+------------+-------------------+-----------------+------------+
|         1|         122| 0.7786885245901639|319.1378119803585|       36.42|
|         0|      631864|                0.5|98.55803220465069|         1.1|
+----------+------------+-------------------+-----------------+------------+



Bisecting for customer behavior

In [16]:
from pyspark.ml.clustering import BisectingKMeans

for k in range(2, 11):
    bkmeans = BisectingKMeans(featuresCol="features", k=k, seed=42)
    bk_model = bkmeans.fit(final_df)
    bk_predictions = bk_model.transform(final_df)
    num_clusters = bk_predictions.select("prediction").distinct().count()
    bk_silhouette = evaluator.evaluate(bk_predictions)
    print(f"k={k}, num_clusters={num_clusters}, score={bk_silhouette}")


k=2, num_clusters=2, score=0.36322020559669516
k=3, num_clusters=3, score=0.5241748315127917
k=4, num_clusters=4, score=0.7143680029329005
k=5, num_clusters=5, score=0.7086121700322872
k=6, num_clusters=6, score=0.7307135504846527
k=7, num_clusters=7, score=0.6784686227541574
k=8, num_clusters=8, score=0.6429572240273124
k=9, num_clusters=9, score=0.6241462159514466
k=10, num_clusters=10, score=0.6043099069633028


In [17]:
# select k=6
bkmeans_behav = BisectingKMeans(featuresCol="features", k=6, seed=42)
bk_model_behav = bkmeans_behav.fit(final_df)
bk_predictions_behav = bk_model_behav.transform(final_df)

In [18]:
profile_bk_behav = bk_predictions_behav.groupBy("prediction").agg(
    F.count("user_id").alias("Cluster_Size"),
    F.avg("recency_days").alias("Avg_Recency"),
    F.avg("verified_rate").alias("Avg_Verified_Rate"),
    F.round(F.avg("is_summer_shopper"),2).alias("Pct_Summer_Shoppers"),
    F.round(F.avg("is_winter_shopper"),2).alias("Pct_Winter_Shoppers"),
    F.round(F.avg("is_autumn_shopper"),2).alias("Pct_Autumn_Shoppers"),
    F.round(F.avg("is_spring_shopper"),2).alias("Pct_Spring_Shoppers"),
)
profile_bk_behav.show()

+----------+------------+------------------+--------------------+-------------------+-------------------+-------------------+-------------------+
|prediction|Cluster_Size|       Avg_Recency|   Avg_Verified_Rate|Pct_Summer_Shoppers|Pct_Winter_Shoppers|Pct_Autumn_Shoppers|Pct_Spring_Shoppers|
+----------+------------+------------------+--------------------+-------------------+-------------------+-------------------+-------------------+
|         1|      143621|1676.5890364222503|  0.9997698111229557|                1.0|                0.0|                0.0|                0.0|
|         3|      148763|1708.4974019077324|  0.9991376664000231|                0.0|                1.0|                0.0|                0.0|
|         5|      153610|1730.7077273614998|  0.9295269123385781|                0.0|                0.0|                0.0|                1.0|
|         4|      138525|1729.1879949467605|  0.9147125083102343|                0.0|                0.0|                1.0

Bisecting for customer loyalty

In [19]:
for k in range(2, 11):
    bkmeans = BisectingKMeans(featuresCol="features", k=k, seed=42)
    bk_model = bkmeans.fit(user_final_df)
    bk_predictions = bk_model.transform(user_final_df)
    num_clusters = bk_predictions.select("prediction").distinct().count()
    bk_silhouette = evaluator.evaluate(bk_predictions)
    print(f"k={k}, num_clusters={num_clusters}, score={bk_silhouette}")

k=2, num_clusters=2, score=0.6038572104125205
k=3, num_clusters=3, score=0.634992628669193
k=4, num_clusters=4, score=0.7094524178896464
k=5, num_clusters=5, score=0.5304836237888955
k=6, num_clusters=6, score=0.5568850753232805
k=7, num_clusters=7, score=0.5902675221277207
k=8, num_clusters=8, score=0.6029261894518976
k=9, num_clusters=9, score=0.38555956537691566
k=10, num_clusters=10, score=0.35858739309104626


In [20]:
# select k=4
bkmeans_user = BisectingKMeans(featuresCol="features", k=4, seed=42)
bk_model_user = bkmeans_user.fit(user_final_df)
bk_predictions_user = bk_model_user.transform(user_final_df)


In [21]:
profile_bk_user = bk_predictions_user.groupBy("prediction").agg(
    F.count("user_id").alias("Cluster_Size"),
    F.avg("sentiment_score").alias("Avg_sentiment_score"),
    F.avg("avg_review_length").alias("Avg_review_length"),
    F.round(F.avg("total_reviews"),2).alias("total_review")
)
profile_bk_user.show()

+----------+------------+-------------------+------------------+------------+
|prediction|Cluster_Size|Avg_sentiment_score| Avg_review_length|total_review|
+----------+------------+-------------------+------------------+------------+
|         1|       16212|-0.5890698248211201|387.91947761196525|        1.44|
|         3|       31187|                1.0| 453.3342938746785|        1.37|
|         2|      414102|                1.0| 70.48798408283008|        1.08|
|         0|      170485|-0.7021849429568584| 74.48106810122941|         1.1|
+----------+------------+-------------------+------------------+------------+



26/03/12 10:57:31 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 835453 ms exceeds timeout 120000 ms
26/03/12 10:57:31 WARN SparkContext: Killing executors is not supported by current scheduler.
26/03/12 10:57:40 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$